In [3]:

import requests, json, urllib.parse, time

TOKEN = "skZFMehj3STc5EGpVcQPUP5PQRmE4kWEQps0Zso4Rl5Ri3QUfmKRViMkpQ6lkHXZTrnHn0kuQgj6y6x7b6Y0Uz0z1jXPmYCKXVbAvYeZcSFOD7mk6uTEeE3MRSLTanEaUjtrPVEO6DkRdKAt6MOHv0zU4NgWek5XVMcahI6TvYOzLqORIR9J"
SAN_HDR  = {"Authorization": f"Bearer {TOKEN}", "Content-Type": "application/json"}
BASE_URL = "https://nzcwegq7.api.sanity.io/v2021-06-07/data"

def query(q):
    r = requests.get(f"{BASE_URL}/query/production?query={urllib.parse.quote(q)}",
                     headers={"Authorization": f"Bearer {TOKEN}"}, timeout=30)
    return r.json().get("result", [])

def mutate(mutations):
    r = requests.post(f"{BASE_URL}/mutate/production",
                      json={"mutations": mutations}, headers=SAN_HDR, timeout=60)
    return r.json()

# Pobierz WSZYSTKIE kategorie
cats = []
offset = 0
while True:
    batch = query(f'*[_type=="category"][{offset}..{offset+499}]')
    cats.extend(batch)
    if len(batch) < 500:
        break
    offset += 500

print(f"Kategorii łącznie: {len(cats)}")

# Analiza pól
has_parent_cat = [c for c in cats if c.get('parentCategory')]
has_parent     = [c for c in cats if c.get('parent')]
has_title      = [c for c in cats if c.get('title') and not c.get('name')]
has_name       = [c for c in cats if c.get('name')]

print(f"  ma 'parentCategory' (błędne): {len(has_parent_cat)}")
print(f"  ma 'parent' (poprawne):       {len(has_parent)}")
print(f"  ma 'title' bez 'name':        {len(has_title)}")
print(f"  ma 'name':                    {len(has_name)}")


Kategorii łącznie: 557
  ma 'parentCategory' (błędne): 256
  ma 'parent' (poprawne):       285
  ma 'title' bez 'name':        264
  ma 'name':                    293


In [7]:

mutations = []

for c in cats:
    cid = c['_id']
    patch_set = {}
    patch_unset = []
    
    # title → name
    if c.get('title') and not c.get('name'):
        patch_set['name'] = c['title']
        patch_unset.append('title')
    
    # parentCategory → parent
    if c.get('parentCategory') and not c.get('parent'):
        patch_set['parent'] = c['parentCategory']  # to już jest {_type, _ref}
        patch_unset.append('parentCategory')
    elif c.get('parentCategory') and c.get('parent'):
        # mamy oba — usuń parentCategory
        patch_unset.append('parentCategory')
    
    if patch_set or patch_unset:
        mutation = {"patch": {"id": cid}}
        if patch_set:
            mutation["patch"]["set"] = patch_set
        if patch_unset:
            mutation["patch"]["unset"] = patch_unset
        mutations.append(mutation)

print(f"Mutacji do wysłania: {len(mutations)}")
print("Przykład:", json.dumps(mutations[0], indent=2, ensure_ascii=False)[:300])


Mutacji do wysłania: 264
Przykład: {
  "patch": {
    "id": "cat-aczniki-do-izolacji-fasadowych",
    "set": {
      "name": "Łączniki do izolacji fasadowych",
      "parent": {
        "_ref": "cat-akcesoria-do-izolacji",
        "_type": "reference"
      }
    },
    "unset": [
      "title",
      "parentCategory"
    ]
  }
}


In [11]:

# Wyślij w paczkach po 200
fixed = 0
for i in range(0, len(mutations), 200):
    batch = mutations[i:i+200]
    res = mutate(batch)
    fixed += len(batch)
    print(f"  Wysłano {fixed}/{len(mutations)}", flush=True)

print(f"\n✅ Naprawiono {fixed} kategorii")

# Weryfikacja po naprawie
time.sleep(2)
top_level = query('count(*[_type=="category" && !defined(parent)])')
print(f"Top-level kategorii po naprawie: {top_level}")


  Wysłano 200/264


  Wysłano 264/264



✅ Naprawiono 264 kategorii


Top-level kategorii po naprawie: 16


In [15]:

import requests, json, urllib.parse, time

TOKEN = "skZFMehj3STc5EGpVcQPUP5PQRmE4kWEQps0Zso4Rl5Ri3QUfmKRViMkpQ6lkHXZTrnHn0kuQgj6y6x7b6Y0Uz0z1jXPmYCKXVbAvYeZcSFOD7mk6uTEeE3MRSLTanEaUjtrPVEO6DkRdKAt6MOHv0zU4NgWek5XVMcahI6TvYOzLqORIR9J"
SAN_HDR = {"Authorization": f"Bearer {TOKEN}", "Content-Type": "application/json"}

def query(q):
    r = requests.get(
        f"https://nzcwegq7.api.sanity.io/v2021-06-07/data/query/production?query={urllib.parse.quote(q)}",
        headers={"Authorization": f"Bearer {TOKEN}"}, timeout=30)
    return r.json().get("result", [])

def mutate(mutations):
    r = requests.post("https://nzcwegq7.api.sanity.io/v2021-06-07/data/mutate/production",
                      json={"mutations": mutations}, headers=SAN_HDR, timeout=60)
    return r.json()

# Pobierz wszystkie top-level kategorie
top = query('*[_type=="category" && !defined(parent)]{_id, name, slug, _createdAt}')
print(f"Top-level kategorii: {len(top)}")
for c in sorted(top, key=lambda x: x.get('name','') or ''):
    print(f"  {c['_id']:50s} | {c.get('name','')}")


Top-level kategorii: 16
  cat-chemia                                         | Chemia budowlana
  cat-chemia-budowlana                               | Chemia budowlana
  cat-dachy                                          | Dachy
  cat-farby                                          | Farby i rozpuszczalniki
  cat-farby-i-rozpuszczalniki                        | Farby i rozpuszczalniki
  cat-izolacje                                       | Izolacje
  cat-narzedzia                                      | Narzędzia i mocowania
  cat-narzedzia-i-mocowania                          | Narzędzia i mocowania
  cat-pozostale                                      | Pozostałe
  cat-plytki                                         | Płytki
  cat-pytki                                          | Płytki
  cat-stropy                                         | Stropy i ściany
  cat-stropy-i-sciany                                | Stropy i ściany
  cat-sucha                                          | Sucha zab

In [19]:

# Stare duplikaty do usunięcia (krótkie slugi - przed reimportem, bez produktów)
# Nowe poprawne: cat-chemia-budowlana, cat-farby-i-rozpuszczalniki, etc.
DELETE_IDS = [
    "cat-chemia",           # duplikat cat-chemia-budowlana
    "cat-farby",            # duplikat cat-farby-i-rozpuszczalniki
    "cat-narzedzia",        # duplikat cat-narzedzia-i-mocowania
    "cat-stropy",           # duplikat cat-stropy-i-sciany
    "cat-sucha",            # duplikat cat-sucha-zabudowa
    "cat-pytki",            # literówka (prawidłowe: cat-plytki)
]

# Sprawdź czy stare mają dzieci
for cat_id in DELETE_IDS:
    children = query(f'count(*[_type=="category" && parent._ref == "{cat_id}"])')
    products  = query(f'count(*[_type=="product" && category._ref == "{cat_id}"])')
    print(f"{cat_id}: {children} dzieci, {products} produktów")

print()
# Sprawdź cat-plytki osobno
plytki_prod = query('count(*[_type=="product" && category._ref == "cat-plytki"])')
print(f"cat-plytki (zostaje): {plytki_prod} produktów")


cat-chemia: 10 dzieci, 0 produktów


cat-farby: 7 dzieci, 0 produktów


cat-narzedzia: 11 dzieci, 0 produktów


cat-stropy: 4 dzieci, 0 produktów


cat-sucha: 7 dzieci, 0 produktów


cat-pytki: 3 dzieci, 0 produktów

cat-plytki (zostaje): 0 produktów


In [23]:

# Mapowanie stare → nowe
REMAP = {
    "cat-chemia":    "cat-chemia-budowlana",
    "cat-farby":     "cat-farby-i-rozpuszczalniki",
    "cat-narzedzia": "cat-narzedzia-i-mocowania",
    "cat-stropy":    "cat-stropy-i-sciany",
    "cat-sucha":     "cat-sucha-zabudowa",
    "cat-pytki":     "cat-plytki",
}

# 1. Przepnij dzieci
for old_id, new_id in REMAP.items():
    children = query(f'*[_type=="category" && parent._ref == "{old_id}"]{{_id}}')
    if children:
        muts = [{"patch": {"id": c["_id"], "set": {"parent": {"_type": "reference", "_ref": new_id}}}} for c in children]
        mutate(muts)
        print(f"  {old_id} → {new_id}: przepięto {len(children)} dzieci")

# 2. Usuń stare duplikaty
del_muts = [{"delete": {"id": cid}} for cid in REMAP.keys()]
mutate(del_muts)
print(f"\n✅ Usunięto {len(del_muts)} starych kategorii")

import time; time.sleep(2)
top_after = query('count(*[_type=="category" && !defined(parent)])')
print(f"Top-level po naprawie: {top_after}")


  cat-chemia → cat-chemia-budowlana: przepięto 10 dzieci


  cat-farby → cat-farby-i-rozpuszczalniki: przepięto 7 dzieci


  cat-narzedzia → cat-narzedzia-i-mocowania: przepięto 11 dzieci


  cat-stropy → cat-stropy-i-sciany: przepięto 4 dzieci


  cat-sucha → cat-sucha-zabudowa: przepięto 7 dzieci


  cat-pytki → cat-plytki: przepięto 3 dzieci



✅ Usunięto 6 starych kategorii


Top-level po naprawie: 10
